**Transform Drivers data**
1. Read bronze drivers table
2. Keep only the columns required for analytics. (Drop url column)
3. Standardise column names using snake_case (driverId driver_id, date0fbirth date_of_birth)
4. Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case
5. Remove duplicate records
6. Transform values of column nationality to Title Case
7. Write the transformed data to silver drivers table


In [0]:
%run "/Workspace/Users/ganeshgadade157@gmail.com/Projects/Formula-f1/00.common/01.envioment_config"

In [0]:
from pyspark.sql import functions as f

folder_name = 'drivers/drivers_flat'
file_name = 'drivers'
silver_table = f"dev.silver.{file_name}"

# 1. Read Bronze
raw_df = spark.read.format('parquet')\
              .load(f"{bronze_path}/{folder_name}")

# 2. Cast types + Drop url
raw_df = raw_df.withColumn('dateOfBirth', f.col('dateOfBirth').cast('date'))\
               .withColumn('ingestion_timestamp', f.col('ingestion_timestamp').cast('timestamp'))\
               .withColumn('driver_name', f.concat(f.col('givenName'), f.lit(' '), f.col('familyName')))\
               .drop('url')

# 3. Rename columns
raw_df = raw_df.withColumnRenamed('driverId', 'driver_id')\
               .withColumnRenamed('dateOfBirth', 'date_of_birth')

# 4. Filter null business key
raw_df = raw_df.filter(f.col('driver_id').isNotNull())

# 5. Remove duplicates
raw_df = raw_df.dropDuplicates(['driver_id'])

# 6. Title Case
raw_df = raw_df.withColumn('nationality', f.initcap(f.col('nationality')))

# 7. Write to Silver
raw_df.write\
      .format('delta')\
      .mode('overwrite')\
      .saveAsTable(silver_table)